In [1]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor


import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []

import importlib 
import eval_utils
importlib.reload(eval_utils)
from eval_utils import load_model
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-10-29/no_encode_intensity_concat_comp_concat_neg_mask_mp_20")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


  0%|          | 0/9046 [00:00<?, ?it/s]

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/

CrystDataModule(self.datasets={'train': {'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy train', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/train.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}, 'val': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy val', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/val.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}], 'test': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy test', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/test.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}]}, self

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [2]:
model.to('cuda:0')

CDVAE(
  (encoder): DimeNetPlusPlusWrap(
    (rbf): BesselBasisLayer(
      (envelope): Envelope()
    )
    (sbf): SphericalBasisLayer(
      (envelope): Envelope()
    )
    (emb): EmbeddingBlock(
      (emb): Embedding(95, 128)
      (lin_rbf): Linear(in_features=6, out_features=128, bias=True)
      (lin): Linear(in_features=384, out_features=128, bias=True)
    )
    (output_blocks): ModuleList(
      (0): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin_up): Linear(in_features=128, out_features=256, bias=True)
        (lins): ModuleList(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Linear(in_features=256, out_features=256, bias=True)
          (2): Linear(in_features=256, out_features=256, bias=True)
        )
        (lin): Linear(in_features=256, out_features=256, bias=False)
      )
      (1): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin

In [7]:
for idx, batch in enumerate(loader):
    print(idx)
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

idx = list_of_idxs[0]
batch = list_of_batchs[0]

def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, batch

batch_reserve, xrd_int, xrd_loc, atom_spec, batch = new_dataloader_batch_processor(batch)

batch_all_frac_coords = []
batch_all_atom_types = []
batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
batch_lengths, batch_angles = [], []

#wrecking the batch value to demonstrate decoding only from the xrd info
_, _, z = model.encode(None, xrd_int, xrd_loc, atom_spec)

self = model

num_atoms = batch.num_atoms.to('cuda:0')
atom_spec = atom_spec.to('cuda:0')
z = z.to('cuda:0')

#number of steps set unusually low to get a rough estimate of performance
ld_kwargs = SimpleNamespace(n_step_each=10,
                                step_lr=1e-4,
                                min_sigma=0,
                                save_traj=False,
                                disable_bar=False)

force_num_atoms = True
gt_num_atoms = num_atoms if force_num_atoms else None

gt_atom_types = None

outputs = model.langevin_dynamics(
    z, ld_kwargs, gt_num_atoms, gt_atom_types, atom_spec)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35


/home/gridsan/tmackey/cdvae/cdvae/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
100%|██████████| 50/50 [01:39<00:00,  1.99s/it]


In [8]:
#manually code up the structure matcher to reproduce match rates before getting the diffraction pattern stuff
from pymatgen.core.structure import Structure
from pymatgen.core.composition import Composition
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.structure_matcher import StructureMatcher
from matminer.featurizers.site.fingerprint import CrystalNNFingerprint
from matminer.featurizers.composition.composite import ElementProperty

In [22]:
#before looping over all structures, implement the comparison for just two 
#create a pymatgen crystal object for the first crystal 
pred_lengths = outputs['lengths'][0].cpu()
pred_angles = outputs['angles'][0].cpu()

pred_lattice = Lattice.from_parameters(pred_lengths[0], pred_lengths[1], pred_lengths[2], 
                                       pred_angles[0], pred_angles[1], pred_angles[2])
pred_num_atoms = outputs['num_atoms'][0].cpu()
outputs['atom_types'] = outputs['atom_types'].cpu()
outputs['frac_coords'] = outputs['frac_coords'].cpu()
predicted_structure = Structure(lattice=pred_lattice, species=outputs['atom_types'][range(0, pred_num_atoms)], 
                                coords=outputs['frac_coords'][range(0, pred_num_atoms)], coords_are_cartesian=False)

In [23]:
predicted_structure

Structure Summary
Lattice
    abc : 4.020460945367681 5.199723437863849 11.21370792388916
 angles : 88.08203071587549 84.81517159062459 88.89629241098734
 volume : 233.3047288432778
      A : 4.0040106773376465 0.0 0.36332452297210693
      B : 0.08477800339460373 5.1961188316345215 0.17402760684490204
      C : 0.0 0.0 11.21370792388916
    pbc : True True True
PeriodicSite: Ga (2.652, 3.169, 8.739) [0.6493, 0.6099, 0.7488]
PeriodicSite: Ga (2.635, 3.129, 3.142) [0.6453, 0.6022, 0.2499]
PeriodicSite: Te (2.646, 2.09, 0.3115) [0.6523, 0.4022, 0.0004017]
PeriodicSite: Ga (0.5988, 0.5888, 5.664) [0.1472, 0.1133, 0.4985]
PeriodicSite: Ga (0.6272, 0.5692, 11.25) [0.1543, 0.1095, 0.9962]
PeriodicSite: Te (0.6961, 4.623, 8.551) [0.155, 0.8896, 0.7437]
PeriodicSite: Te (0.663, 4.67, 3.016) [0.1465, 0.8988, 0.2503]
PeriodicSite: Te (2.613, 2.018, 5.918) [0.6443, 0.3884, 0.5008]

In [25]:
#make a crystal structure for the ground truth
gt_lengths = batch['lengths'][0].cpu()
gt_angles = batch['angles'][0].cpu()

pred_lattice = Lattice.from_parameters(gt_lengths[0], gt_lengths[1], gt_lengths[2], 
                                       gt_angles[0], gt_angles[1], gt_angles[2])
gt_num_atoms = batch['num_atoms'][0].cpu()
batch['atom_types'] = batch['atom_types'].cpu()
batch['frac_coords'] = batch['frac_coords'].cpu()
gt_structure = Structure(lattice=pred_lattice, species=batch['atom_types'][range(0, gt_num_atoms)], 
                                coords=batch['frac_coords'][range(0, gt_num_atoms)], coords_are_cartesian=False)

In [26]:
gt_structure

Structure Summary
Lattice
    abc : 4.134600162506107 4.134599673861435 18.425569534301758
 angles : 90.00000250447802 90.00000250447812 120.00000390950429
 volume : 272.78376468258256
      A : 4.1346001625061035 0.0 -1.8072911700528493e-07
      B : -2.0673000812530518 3.5806682109832764 -1.807290885835755e-07
      C : 0.0 0.0 18.425569534301758
    pbc : True True True
PeriodicSite: Ga (2.067, 1.194, 15.05) [0.6667, 0.3333, 0.817]
PeriodicSite: Ga (0.0, 2.387, 5.842) [0.3333, 0.6667, 0.317]
PeriodicSite: Ga (0.0, 2.387, 3.371) [0.3333, 0.6667, 0.183]
PeriodicSite: Ga (2.067, 1.194, 12.58) [0.6667, 0.3333, 0.683]
PeriodicSite: Te (2.067, 1.194, 2.097) [0.6667, 0.3333, 0.1138]
PeriodicSite: Te (0.0, 2.387, 11.31) [0.3333, 0.6667, 0.6138]
PeriodicSite: Te (0.0, 2.387, 16.33) [0.3333, 0.6667, 0.8862]
PeriodicSite: Te (2.067, 1.194, 7.116) [0.6667, 0.3333, 0.3862]

In [27]:
#get the first structure in the output and compare it to the first structure in the batch by constructing pymatgen crystal objects

stol=0.5
angle_tol=10
ltol=0.3
matcher = StructureMatcher(stol=stol, angle_tol=angle_tol, ltol=ltol)
rms_dist = matcher.get_rms_dist(predicted_structure, gt_structure)

In [28]:
rms_dist
#the first RMS value is for the fractional coordinates, the second RMS value is for cartesian coordinates 

(3.6233453571904236e-16, 9.58647072922636e-16)

In [ ]:
#plot the diffraction patterns for the two structures 

